# Lab 6B: Inverted Pendulum Control

Run the cells in order. The MicroPython program remains active and waits for new experiment commands after the serial connection is established.


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
from serial import Serial
from serial.tools import list_ports
import time
import numpy as np
from matplotlib import pyplot as plt


## 1. Open the Serial Connection

Keep the USB cable connected to the Shoe of Brian. In Thonny, record the serial port shown for `MicroPython (generic)`, then switch the interpreter to `Local Python 3` so that Jupyter can use the same port.


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
for port in list_ports.comports():
    print(port.device, port.description)


Set `SERIAL_PORT` to the Shoe USB port found above. This is the only value to change in the serial-connection setup.


In [ ]:
# TODO: Enter the Shoe USB serial port.
SERIAL_PORT = "COM12"

# PROVIDED VALUE — DO NOT MODIFY.
BAUDRATE = 115200


Open the serial port, restart the MicroPython program, and wait until `main.py` prints `READY LAB6B_SERIAL_V1`.

**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
ser = Serial(SERIAL_PORT, baudrate=BAUDRATE, timeout=0.2)
time.sleep(0.3)
ser.reset_input_buffer()

# Ctrl-B returns to the normal REPL, Ctrl-C stops a running program,
# and Ctrl-D soft-resets MicroPython so main.py runs again.
ser.write(b"\x02")
time.sleep(0.1)
ser.write(b"\x03")
time.sleep(0.1)
ser.write(b"\x04")

deadline = time.time() + 5
ready = False

while time.time() < deadline and ready == False:
    line = ser.readline().decode().strip()

    if line == "READY LAB6B_SERIAL_V1":
        ready = True
        print(line)

if ready == False:
    print("ERR did not receive READY. Check that main.py is uploaded and has no errors.")
else:
    print("Serial port is open and main.py is running.")


## 2. Define `run_command()`


**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
def run_command(command, timeout=20):
    lines = []
    ser.write((command + "\r\n").encode())

    deadline = time.time() + timeout
    finished = False

    while time.time() < deadline and finished == False:
        current_line = ser.readline().decode().strip()

        if current_line != "":
            lines.append(current_line)

            if current_line == "END":
                finished = True
            elif current_line.startswith("DATA,") == False:
                print(current_line)

    if finished == False:
        print("ERR no END before timeout")

    return lines


## 3. Define the Experiment and Plotting Functions


**Provided code — do not modify this cell.**


In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
def plot_response(filename, pulse_width, kp, ki, kd, setpoint_rad, switch_angle):
    data = np.genfromtxt(filename, delimiter=",")
    data = np.atleast_2d(data)

    times_s = data[:, 0]
    positions_rad = data[:, 1]
    outputs_percent = data[:, 2]
    p_actions = data[:, 3]
    i_actions = data[:, 4]
    d_actions = data[:, 5]

    fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

    axes[0].plot(times_s, positions_rad, label="Measured position")
    axes[0].axhline(setpoint_rad, color="k", linestyle="--", label="Setpoint")
    axes[0].set_ylabel("Position [rad]")
    axes[0].set_title(f"Swing-Up Response: Kp={kp:g}, Ki={ki:g}, Kd={kd:g}")
    axes[0].grid(True)
    axes[0].legend()

    axes[1].plot(times_s, p_actions, "--", label="P action")
    axes[1].plot(times_s, i_actions, "--", label="I action")
    axes[1].plot(times_s, d_actions, "--", label="D action")
    axes[1].plot(times_s, outputs_percent, label="Total output")
    axes[1].set_xlabel("Time [s]")
    axes[1].set_ylabel("Voltage command [%]")
    axes[1].set_ylim(-110, 110)
    axes[1].grid(True)
    axes[1].legend()

    fig.tight_layout()
    plt.show()


def run_experiment(
    pulse_width,
    kp,
    ki,
    kd,
    setpoint_rad,
    test_time_s,
    output_limit_percent,
    switch_angle,
    data_filename,
):
    encoder_init = 0 # default initialization of encoder in downward pendulum position
    response_lines = run_command(
        (
            f"RUN {pulse_width} {kp} {ki} {kd} {setpoint_rad} "
            f"{test_time_s} {output_limit_percent} {encoder_init} {switch_angle}"
        ),
        timeout=test_time_s + 10,
    )

    data_lines = [
        line[len("DATA,"):]
        for line in response_lines
        if line.startswith("DATA,")
    ]

    if len(data_lines) == 0:
        print("ERR no data were returned by the microcontroller")
        return

    with open(data_filename, "w") as data_file:
        for data_line in data_lines:
            data_file.write(data_line + "\n")

    print(f"Saved {len(data_lines)} samples to {data_filename}.")
    plot_response(data_filename, pulse_width, kp, ki, kd, setpoint_rad, switch_angle)


## 4. Run the Experiments

Each experiment cell should contain one call to `run_pid_experiment()`. 

In [ ]:
# TODO: Duplicate this cell for each experiment and enter its parameters.
run_experiment(
    pulse_width=0.0,
    kp=0.0,
    ki=0.0,
    kd=0.0,
    setpoint_rad=0.0,
    test_time_s=5.0,
    output_limit_percent=100.0,
    switch_angle=0.0,
    data_filename="swingup.csv",
)

## 5. Close the Serial Connection



In [ ]:
# PROVIDED CODE — DO NOT MODIFY THIS CELL.
ser.close()
